In [7]:
from pathlib import Path

import time
import pandas as pd
import polars as pl

In [5]:
BASE_PATH = Path.cwd().parent / "titanic-competition"
TRAIN_PATH = BASE_PATH / "train.csv"

## Comparison: Pandas (Eager) vs. Polars (Lazy)

| Feature | Pandas (read_csv) | Polars (scan_csv) |
|---|---|---|
| Execution Style | Eager: Executes every line immediately and sequentially. | Lazy: Delays execution until you explicitly call .collect(). |
| Memory Blueprint | Loads all 12 columns and rows into RAM first. | Allocates zero data memory until the final trigger. |
| Column Efficiency | Parses and stores unwanted columns (e.g., Name, Cabin) before dropping them. | Projection Pushdown: Reads only the Age and Fare columns directly from disk. |
| Row Efficiency | Allocates space for all 891 rows, then filters them down. | Predicate Pushdown: Filters out rows while reading, never loading excluded rows into RAM. |
| Engine Mindset | Imperative: Follows instructions blindly step-by-step. | Declarative: Optimizes the entire query chain globally before running. |



In [ ]:
# 1. PANDAS (Eager)
start_pandas = time.perf_counter()
df_pd = pd.read_csv(TRAIN_PATH)
# Emulate your pipeline filtering/selecting
df_pd = df_pd[df_pd["Age"] > 18][["Age", "Fare"]]
end_pandas = time.perf_counter()
pandas_time = end_pandas - start_pandas

# 2. POLARS EAGER
start_pl_eager = time.perf_counter()
df_pl_eager = pl.read_csv(TRAIN_PATH)
df_pl_eager = df_pl_eager.filter(pl.col("Age") > 18).select(["Age", "Fare"])
end_pl_eager = time.perf_counter()
pl_eager_time = end_pl_eager - start_pl_eager

# 3. POLARS LAZY (The Smart Way)
start_pl_lazy_plan = time.perf_counter()
# This only builds the recipe book (Instant)
lazy_plan = (
    pl.scan_csv(TRAIN_PATH)
    .filter(pl.col("Age") > 18)
    .select(["Age", "Fare"])
)
end_pl_lazy_plan = time.perf_counter()

start_pl_lazy_exec = time.perf_counter()
# This triggers the actual optimized loading and processing
df_pl_lazy = lazy_plan.collect()
end_pl_lazy_exec = time.perf_counter()

pl_lazy_total = (end_pl_lazy_plan - start_pl_lazy_plan) + (end_pl_lazy_exec - start_pl_lazy_exec)

# Print the Engineering Results
print(f"🐼 Pandas Eager Time:      {pandas_time:.6f} seconds")
print(f"🐻 Polars Eager Time:      {pl_eager_time:.6f} seconds")
print(f"⚡ Polars Lazy (Plan Only): {end_pl_lazy_plan - start_pl_lazy_plan:.6f} seconds")
print(f"🚀 Polars Lazy (Execution): {end_pl_lazy_exec - start_pl_lazy_exec:.6f} seconds")
print(f"🏆 Polars Lazy Total Time: {pl_lazy_total:.6f} seconds")


🐼 Pandas Eager Time:      0.025572 seconds
🐻 Polars Eager Time:      0.227700 seconds
⚡ Polars Lazy (Plan Only): 0.005787 seconds
🚀 Polars Lazy (Execution): 0.219254 seconds
🏆 Polars Lazy Total Time: 0.225041 seconds


| Dataset Size | Rows | Pandas Eager | Polars Lazy | The Winner |
|---|---|---|---|---|
| Titanic (Small) | 891 | ~0.02s | ~0.22s | Pandas (Polars overhead dominates) |
| Medium Data | 1,000,000 | ~2.5s | ~0.3s | Polars (Optimizations overtake setup) |
| Big Data | 100,000,000 | Out of Memory (Crash) | ~25s | Polars (Queries stream without breaking RAM) |

In [19]:
# To prove this point let's scale the dataset and then look at the performance

In [20]:
# Duplicate it to create ~900,000 rows
df_large = pd.concat([df_pd] * 1000, ignore_index=True)
df_large.to_csv(BASE_PATH / "train_large.csv", index=False)

print("Created 'train_large.csv' with ~900k rows. Now re-run your timing script using this new file!")

Created 'train_large.csv' with ~900k rows. Now re-run your timing script using this new file!


In [21]:
target_file = BASE_PATH / "train_large.csv"

# 🐼 PANDAS (Eager)
start_pandas = time.perf_counter()
df_pd = pd.read_csv(target_file)
df_pd = df_pd[df_pd["Age"] > 18][["Age", "Fare"]]
end_pandas = time.perf_counter()
pandas_time = end_pandas - start_pandas

# 🐻 POLARS EAGER
start_pl_eager = time.perf_counter()
df_pl_eager = pl.read_csv(target_file)
df_pl_eager = df_pl_eager.filter(pl.col("Age") > 18).select(["Age", "Fare"])
end_pl_eager = time.perf_counter()
pl_eager_time = end_pl_eager - start_pl_eager

# 🚀 POLARS LAZY
start_pl_lazy_plan = time.perf_counter()
lazy_plan = (
    pl.scan_csv(target_file)
    .filter(pl.col("Age") > 18)
    .select(["Age", "Fare"])
)
end_pl_lazy_plan = time.perf_counter()

start_pl_lazy_exec = time.perf_counter()
df_pl_lazy = lazy_plan.collect()
end_pl_lazy_exec = time.perf_counter()

pl_lazy_total = (end_pl_lazy_plan - start_pl_lazy_plan) + (end_pl_lazy_exec - start_pl_lazy_exec)

# ==========================================
# 3. PRINT RESULTS
# ==========================================
print("📊 BENCHMARK RESULTS (900,000 Rows):")
print(f"🐼 Pandas Eager Time:      {pandas_time:.6f} seconds")
print(f"🐻 Polars Eager Time:      {pl_eager_time:.6f} seconds")
print(f"⚡ Polars Lazy (Plan Only): {end_pl_lazy_plan - start_pl_lazy_plan:.6f} seconds")
print(f"🚀 Polars Lazy (Execution): {end_pl_lazy_exec - start_pl_lazy_exec:.6f} seconds")
print(f"🏆 Polars Lazy Total Time: {pl_lazy_total:.6f} seconds")

📊 BENCHMARK RESULTS (900,000 Rows):
🐼 Pandas Eager Time:      0.306605 seconds
🐻 Polars Eager Time:      0.040516 seconds
⚡ Polars Lazy (Plan Only): 0.008159 seconds
🚀 Polars Lazy (Execution): 0.048882 seconds
🏆 Polars Lazy Total Time: 0.057041 seconds


## Summary (900,000 Rows)

| Engine / Mode | Phase | Time (Seconds) | Speed vs. Pandas |
|---|---|---|---|
| 🐼 Pandas | Eager (Full Execution) | 0.306605 | 1.0x (Baseline) |
| 🐻 Polars Eager | Eager (Full Execution) | 0.040516 | ~7.6x Faster |
| ⚡ Polars Lazy | Plan Only | 0.008159 | Instant (No I/O) |
| 🚀 Polars Lazy | Execution Only | 0.048882 | — |
| 🏆 Polars Lazy | Total (Plan + Exec) | 0.057041 | ~5.4x Faster |


In [22]:
# This was just to showcase that why polars beats pandas in Big Data.
# The actual commands and understanding starts now

In [23]:
lazy = (
    pl.scan_csv(TRAIN_PATH)
      .filter(pl.col("Age") > 18)
      .select(["Age", "Fare"])
)

In [ ]:
lazy
# the Greek letter pi stands for Projection, which is the formal term for selecting specific columns
# This shows the actual workflow of the 

In [ ]:
"""
This image depicts the exact workflow Polars executes from bottom to top when you call .collect().

## What the Image Depicts : 
    It serves as a visual blueprint of your data pipeline. 
    It shows exactly how data will travel through memory.

## The Step-by-Step Runtime Workflow :

* 1. CSV SCAN: The engine hits the disk.
* 2. Column Dropping: It isolates 2 columns.
* 3. Row Skipping: It discards 10 unneeded columns.
* 4. Streaming: Data moves up the pipeline.
* 5. FILTER BY: Row evaluation triggers next.
* 6. Predicate Pushdown: Records matching Age > 18 pass.
* 7. Eviction: Failing rows drop from memory.
* 8. Final Output ($\pi$ 2/2): The final table forms.

"""

In [30]:
# In the memory is the workflow plan to load the data.
# No data is allocated in ram till now and data is still in disk now.

In [31]:
# Now let's add some more operations to our lazy loader

In [37]:
lazy = (
    pl.scan_csv(TRAIN_PATH)
      .filter(pl.col("Age") > 18)
      .filter(pl.col("Fare") > 50)
      .select(["Age", "Fare"])
)

In [38]:
lazy

In [ ]:
# It silently peeks the columns needed for this operation and the selection.
# Founds out that there is only 2 columns : Only loads and filters two columns
# A huge memory save !!!

In [ ]:
# But not order does matter when there are some operation supposed to happen before a transformation .
# for example : 
""" (
    pl.scan_csv("train.csv")
      .with_columns(
          (pl.col("Age") * 2).alias("Age")
      )
      .filter(pl.col("Age") >= 30)
) """

# IF we closely look at this operation , we can clearly see that eh Age column is being doubled and then filtered.
# If polars were to filter first and then double , then that would result in a two complete dataset 

# For example "Age" : 20,25,30
# If filtering done first  and then Doubled :  "Age" : 60
# If filtering done after Doubling  :  "Age" : 40,50,60

# The optimizer is not allowed to reorder operations arbitrarily.
# It can only reorder them when it can prove that the meaning of your query stays the same.

# Also it loads only loads the necessary column : "Age" only , to avoid unnecessary data.
# This is called Projection Pushdown.
# But because of the csv format itself what it is doing behind the scenes is basically looking at all the rows one by one and remove every other cols.

# TO read only the specific column we need the data in parquet format .
# using : pl.scan_parquet("train.parquet").select("Age")

# Why parquet ? 
# How parquet stores : 
""" 
Age
22
38
26
...

Fare
7.25
71.28
7.92
...

Name
John
Alice
Bob
...
"""
# It stores the data column by column instead of row by row.
# So here we do we fetch the sample from Age only and no other column

In [42]:
lazy.collect_schema() # Load the column names and the datatypes without even loading the dataset .

Schema([('Age', Float64), ('Fare', Float64)])